# GRPO on Colab — Grounded RAG for Mental Health

CLAUDE.md Act 2. Runs end-to-end on a Colab T4:
1. **`prepare_grpo_prompts.py`** — retrieve once per question (Cohere Embed + Rerank), dump prompts + passages.
2. **`train_grpo.py`** — TRL `GRPOTrainer` starting from the merged DPO adapter, LoRA on top. Reward = groundedness − λ·copy_penalty (intentionally partial per step 8).

**Watch for collapse.** Reward log lines print `g`, `copy`, `cite_p/r`, `abst` — the Act 2 story is groundedness rising while copy/abstention/citation collapse. Step 10 will repair with `+ μ·helpfulness`.

## 1. Environment setup

**Before running:** in the left sidebar, click the 🔑 icon and add a secret named `COHERE_API_KEY` with your production key.

In [ ]:
# Confirm we're on a GPU runtime. If this prints "NO GPU", go to Runtime → Change runtime type → T4 GPU.
import subprocess
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'NO GPU — switch runtime')

In [ ]:
REPO_URL = 'https://github.com/nightvoyager111/Grounded-RAG-via-RL-for-mental-health.git'
!rm -rf Grounded-RAG-via-RL-for-mental-health
!git clone {REPO_URL}
%cd Grounded-RAG-via-RL-for-mental-health

In [ ]:
# Deps. TRL >=0.11 has GRPOTrainer.
!pip install -q cohere python-dotenv pyyaml transformers accelerate peft 'trl>=0.11' datasets sentencepiece

In [ ]:
from google.colab import userdata
import os
os.environ['COHERE_API_KEY'] = userdata.get('COHERE_API_KEY')
assert os.environ['COHERE_API_KEY'], 'COHERE_API_KEY secret is empty — set it in the 🔑 sidebar'

## 2. Restore the DPO adapter

`configs/grpo.yaml` expects `checkpoints/dpo/` to exist and merges it into the base model before training. If you saved it to Drive after the DPO run, pull it back here. Set `dpo_adapter: null` in the config instead if you want a clean-base GRPO run (weaker final model; cleaner attribution).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p checkpoints
!cp -r /content/drive/MyDrive/grounded-rag-dpo/checkpoints/dpo checkpoints/
!ls checkpoints/dpo

## 3. Config overrides for CUDA

The repo's `configs/generation.yaml` is set to `device: mps` + `dtype: float32` for local Mac. Rewrite it for CUDA + fp16 (only affects this Colab copy).

In [ ]:
import yaml, pathlib

gen_path = pathlib.Path('configs/generation.yaml')
gen = yaml.safe_load(gen_path.read_text())
gen['device'] = 'cuda'
gen['dtype'] = 'float16'
gen_path.write_text(yaml.safe_dump(gen, sort_keys=False))
print(gen_path.read_text())

## 4. Prepare rollout prompts

Retrieval is Cohere-bound, so we run it once and cache to `data/grpo/prompts.jsonl` — GRPO reuses this every step without re-embedding.

Smoke test first (3 questions), then full.

In [ ]:
!python -m src.scripts.prepare_grpo_prompts --limit 3

In [ ]:
# Eyeball one prompt to confirm the chat template + passage format looks right.
import json
with open('data/grpo/prompts.jsonl') as f:
    row = json.loads(next(f))
print('QUESTION:', row['question'])
print('N_PASSAGES:', len(row['retrieved_ids']))
print('PROMPT (first 800 chars):\n', row['prompt'][:800])

In [ ]:
# Full prep — all 25 questions, few minutes of Cohere calls.
!python -m src.scripts.prepare_grpo_prompts

## 5. Train GRPO

Each step: G rollouts × Cohere judge call each. The `[reward step NNNN]` lines are the collapse story — watch for `g` rising while `copy` climbs toward 1.0 or `abst` spikes.

In [ ]:
!python -m src.scripts.train_grpo

## 6. Persist checkpoint + reward trace

Adapter (~50–200MB) and the per-step reward diagnostics both go to Drive so nothing is lost when the runtime disconnects. The reward trace is the raw material for the step-10 write-up.

In [ ]:
!mkdir -p /content/drive/MyDrive/grounded-rag-dpo/checkpoints
!cp -r checkpoints/grpo /content/drive/MyDrive/grounded-rag-dpo/checkpoints/
!mkdir -p /content/drive/MyDrive/grounded-rag-dpo/grpo_artifacts
!cp data/grpo/reward_trace.jsonl /content/drive/MyDrive/grounded-rag-dpo/grpo_artifacts/

In [ ]:
# Quick collapse-check plot: mean reward + components per step from the trace.
import json, collections
buckets = collections.defaultdict(list)
with open('data/grpo/reward_trace.jsonl') as f:
    for line in f:
        d = json.loads(line)
        buckets[d['step']].append(d)

import statistics as st
def _mean(vals):
    v = [x for x in vals if x is not None]
    return st.mean(v) if v else float('nan')

print(f"{'step':>4} {'reward':>7} {'ground':>7} {'copy':>7} {'cite_p':>7} {'cite_r':>7} {'abst':>5}")
for step in sorted(buckets):
    rs = buckets[step]
    print(f"{step:>4d} {_mean([r['reward'] for r in rs]):>7.3f} "
          f"{_mean([r['groundedness'] for r in rs]):>7.3f} "
          f"{_mean([r['copy_rate'] for r in rs]):>7.3f} "
          f"{_mean([r['citation_precision'] for r in rs]):>7.3f} "
          f"{_mean([r['citation_recall'] for r in rs]):>7.3f} "
          f"{_mean([r['abstention'] for r in rs]):>5.2f}")